# Wakhloo, Slatton & Chung (2026) on DANDI:000978

Paper: A. J. Wakhloo, W. Slatton and S. Chung, "Neural population geometry and optimal coding of
tasks with shared latent structure", *Nature Neuroscience* (2026),
doi:[10.1038/s41593-025-02183-y](https://doi.org/10.1038/s41593-025-02183-y). The paper reuses the
data of Tang, Shin and Jadhav (2023).

Dataset: [DANDI:000978](https://dandiarchive.org/dandiset/000978/0.240511.0307) "Single Day
W-Track Learning", version **0.240511.0307** (doi:10.48324/dandi.000978/0.240511.0307). The paper
cites this version (ref. 67, and the README of its code), and it is also the most recent published
version; the dandiset has three published versions, all made on 2024-05-11 within twenty minutes,
with the same nine assets and the same total size, so the choice does not change the data read.

Claims: `claims/wakhloo-2026.yaml`, written from the version of record (published 4 February 2026),
main text, figure legends and Methods; the Supplementary Information was not in the text read. The
paper's code is github.com/awakhloo/population_geometry_optimal_coding at commit 9674a6c, which the
claims file and the helper `wtrack.py` cite for each method detail the Methods leave open.

| Claim | Tested here |
|---|---|
| eq1-generalization-error-formula, eq1-sample-dependence, optimal-code-disentangled, eq2-optimal-spectrum-flattens, fig6ce-optimal-early-low-dim-high-corr | Blocked: theory, no recorded data |
| fig3bc-mlp-theory-tracks-error, fig3dg-mlp-geometry-tradeoff | Blocked: simulations, no recorded data |
| fig4bf-pose-network-geometry, fig4g-pose-theory-r2 | Blocked: DeepLabCut data elsewhere, needs a trained network |
| fig5d, fig5e, fig5h (macaque) | Blocked: Brain-Score data elsewhere, not attempted in this run |
| svc-hebbian-correlation | Tested (PFC and CA1 part) |
| fig7d-behavioral-error-learning | Tested |
| fig7e-readout-error-learning | Tested |
| figs12c-behavior-readout-r2 | Tested |
| fig7f-theory-vs-empirical | Tested |
| fig7g-total-dimension-drops-early | Tested |
| fig7h-task-dimension-increases | Tested |
| fig7hk-measures-increase-early | Tested |
| fig7-late-reversal | Tested |
| mixed-quadratic-dim-corr-snf | Tested |
| mixed-quadratic-ssf-not-nonmonotonic | Tested |
| mixed-quadratic-task-dim | Tested |
| mixed-linear-ssf-task-dim | Tested |
| late-trends-match-optimal | Tested |
| glm-cross-validated-d2 | Tested |

In [ ]:
import os
for v in ["OMP_NUM_THREADS", "OPENBLAS_NUM_THREADS", "MKL_NUM_THREADS"]:
    os.environ[v] = "1"   # many small fits; one linear-algebra thread each
import time, resource
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from joblib import Parallel, delayed
from statsmodels.stats.multitest import multipletests
import wtrack as w

T0 = time.time()
N_SUBSETS = 500     # Methods: every value is the mean over 500 random subsets
SVC_EVERY = 125     # the SVC is run on every 125th subset (4 per session), see the SVC section
N_PR = 500          # code: participation ratio of the GLM predictions over 500 unit sets
GLM_MAX_UNITS = None # None: every unit is fitted
N_JOBS = 4
print("dandiset", w.DANDISET, "version", w.VERSION)

In [ ]:
urls = w.asset_urls()
print(len(urls), "assets")
for p in sorted(urls):
    print(" ", p)
t = time.time()
D = {r: w.load_rat(r, urls) for r in w.RATS}
print(f"streamed and binned in {time.time() - t:.0f} s; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")

## Data check

What was loaded, per rat and session, and each check of the description against the values.

In [ ]:
rows = []
for r, d in D.items():
    for k, s in enumerate(d["sessions"]):
        z = s["z"]
        sp = np.linalg.norm(z[:, 3:5], axis=1)
        rows.append(dict(rat=r, session=k + 1, file=s["file"], epoch=s["epoch"], trials=s["n_trials"],
                         minutes=round((s["stop"] - s["start"]) / 60, 1), bins=len(z),
                         bins_moving=int((sp > w.VMIN).sum()), bins_moving_speedcol=int((z[:, 2] > w.VMIN).sum()),
                         frac_correct=round(s["frac_correct"], 3),
                         PFC_units=int((s["loc"] == "PFC").sum()), CA1_units=int((s["loc"] == "CA1").sum()),
                         PFC_kept=w.process_session(s["counts"], z, s["loc"], "PFC")[0].shape[1],
                         CA1_kept=w.process_session(s["counts"], z, s["loc"], "CA1")[0].shape[1],
                         max_gap_s=s["max_gap"]))
S = pd.DataFrame(rows)
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 30); pd.set_option("display.max_rows", 100)
print(S.to_string(index=False))
print("\nrats", S.rat.nunique(), "| sessions per rat", S.groupby("rat").size().unique(), "| session length", S.minutes.min(), "to", S.minutes.max(), "min")
print("fewest units kept in a session: PFC", S.PFC_kept.min(), "CA1", S.CA1_kept.min(), "(Methods use 19 PFC and 24 CA1 units)")
print("fewest moving bins in a session", S.bins_moving.min(), "(the subsets take 500)")
print("largest gap between consecutive trials within a session", S.max_gap_s.max(), "s (the code asserts none)")

print("\nPer file:")
for r, d in D.items():
    for raw in d["raw"]:
        tr = raw["trials"]
        print(f"{raw['path']}: subject {raw['info']['subject_id']}, session_id {raw['info']['session_id']}, units {raw['n_units']} "
              f"({', '.join(f'{k} {int((raw['unit_loc'] == k).sum())}' for k in sorted(set(raw['unit_loc'])))}), "
              f"epochs {len(raw['epochs'])}, trials {len(tr)} (correct values {sorted(set(tr.correct))}, trajectory types {sorted(set(tr.trajectory_type))}), "
              f"spike times sorted {raw['spikes_sorted']}")
        print(f"    position unit '{raw['info']['pos_unit']}', x {raw['pos_range'][0][0]:.1f} to {raw['pos_range'][1][0]:.1f}, "
              f"y {raw['pos_range'][0][1]:.1f} to {raw['pos_range'][1][1]:.1f}, speed column {raw['pos_range'][0][2]:.1f} to {raw['pos_range'][1][2]:.1f}; "
              f"frame interval {raw['pos_dt'] * 1000:.2f} ms ({1 / raw['pos_dt']:.1f} fps); position t {raw['pos_t_range'][0]:.0f} to {raw['pos_t_range'][1]:.0f} s, "
              f"spikes t {raw['spike_t_range'][0]:.0f} to {raw['spike_t_range'][1]:.0f} s; electrical series {raw['info']['ephys_shape']}")

# recordings deposited more than once: identical spike trains or position traces across files
allh = [(raw["path"], u, h) for d in D.values() for raw in d["raw"] for u, h in enumerate(raw["spike_hashes"])]
byh = {}
for p, u, h in allh:
    byh.setdefault(h, []).append((p, u))
dups = [v for v in byh.values() if len(v) > 1 and len({p for p, _ in v}) > 1]
print("\nunit spike trains identical across files:", len(dups))
ph = [raw["pos_hash"] for d in D.values() for raw in d["raw"]]
print("position traces identical across files:", len(ph) - len(set(ph)))
z1, z2 = D["ZT2"]["raw"]
print("ZT2 files: position time ranges", [tuple(np.round(x["pos_t_range"])) for x in (z1, z2)],
      "| trial time ranges", [(round(x["trials"].start_time.min()), round(x["trials"].stop_time.max())) for x in (z1, z2)])

# speed: the dataset's speed column against the norm of the smoothed velocity
sp_col = np.concatenate([s["z"][:, 2] for d in D.values() for s in d["sessions"]])
sp_vel = np.concatenate([np.linalg.norm(s["z"][:, 3:5], axis=1) for d in D.values() for s in d["sessions"]])
print(f"\nspeed column against |smoothed velocity| per bin: r = {np.corrcoef(sp_col, sp_vel)[0, 1]:.3f}, "
      f"median ratio {np.median(sp_col[sp_vel > 4] / sp_vel[sp_vel > 4]):.2f}; bins above 4 cm/s: {np.mean(sp_col > 4):.3f} (column), {np.mean(sp_vel > 4):.3f} (velocity)")

### Clock check

Spikes, position and trials must share one clock. Two contrasts show it:
(1) position from trials: at the end of each trial the animal should be at the trial's end well,
so the position just before the end time should be nearest to that well, and less often so 15 s
earlier; (2) spikes from position: a ridge decoder of x and y position from
CA1 counts (500 ms bins, five contiguous folds) should do better with the true alignment than with
the counts shifted by 10 s or more. The decoder and its penalty are a standard choice for this check
only.

In [ ]:
rows = []
for r, d in D.items():
    for k, s in enumerate(d["sessions"]):
        z = s["z"]
        m = np.linalg.norm(z[:, 3:5], axis=1) > w.VMIN
        x = s["counts"][m][:, s["loc"] == "CA1"]
        x = (x - x.mean(0)) / (x.std(0) + 1e-9)
        y = z[m][:, :2]
        rows.append(dict(rat=r, session=k + 1, r2_true=w.decode_r2(x, y), r2_shift_10s=w.decode_r2(x, y, 20),
                         r2_shift_60s=w.decode_r2(x, y, 120)))
C = pd.DataFrame(rows)
print("position decoded from CA1 counts, cross-validated R2 per session (summary over the 64 sessions):")
print(C[["r2_true", "r2_shift_10s", "r2_shift_60s"]].describe().round(3).to_string())
print("sessions where the true alignment beats both shifts:", int(((C.r2_true > C.r2_shift_10s) & (C.r2_true > C.r2_shift_60s)).sum()), "of", len(C))
print(C.groupby("rat")[["r2_true", "r2_shift_10s"]].mean().round(3).to_string())

# trials against position: the position 0.5 s before each trial's end should be at the trial's end well.
# Well positions are the session's median end positions per end well; each trial is assigned to the
# nearest one, with the true end times and with the end times 15 s earlier.
hit = {0: [], 15: []}
dist = {0: [], 15: []}
for r, d in D.items():
    for k, s in enumerate(d["sessions"]):
        raw = d["raw"][s["file"]]
        tr = raw["trials"][raw["trials"].epoch == s["epoch"]]
        bt = s["start"] + (np.arange(len(s["z"])) + 0.5) * w.DT
        at = lambda t: np.stack([np.interp(t, bt, s["z"][:, 0]), np.interp(t, bt, s["z"][:, 1])], 1)
        wells = np.unique(tr.end_well)
        cen = np.stack([np.median(at(tr.stop_time.values - 0.5)[tr.end_well.values == wl], 0) for wl in wells])
        for sh in hit:
            xy = at(tr.stop_time.values - 0.5 - sh)
            dd = np.linalg.norm(xy[:, None] - cen[None], axis=2)
            hit[sh] += list(wells[dd.argmin(1)] == tr.end_well.values)
            dist[sh] += list(dd[np.arange(len(tr)), np.searchsorted(wells, tr.end_well.values)])
print(f"\ntrials whose position 0.5 s before the end is nearest to their end well: true times {np.mean(hit[0]):.3f}, 15 s earlier {np.mean(hit[15]):.3f}; "
      f"median distance to the end well {np.median(dist[0]):.1f} cm and {np.median(dist[15]):.1f} cm (n = {len(hit[0])} trials)")

## Readout analysis (shared pipeline)

For each area, rat and session: locomotion bins (speed of the smoothed velocity above 4 cm/s), units
above 0.1 Hz, each unit centred and divided by its standard deviation plus 0.01 counts per bin
(0.02 Hz), latents x, y, vx, vy z-scored (Methods, Readout analysis; code `rat_utils.process_dat`).
Then 500 random subsets of units (19 PFC, 24 CA1 as the Methods; 19 CA1 as the code, a variant) and
of 500 bins; per subset the theoretical error (Eq. 5, p = 300), the empirical Hebbian error (300
training and 200 test bins, 500 Gaussian tasks with |mean label| < 0.25), dimension, correlation,
SNF and SSF (Eqs. 6 to 9). Each session's value is the mean over subsets. Each (area, rat, session)
has its own random generator, seeded from its indices.

The linear SVC of the code is fitted on the same split, one per task, which costs about 1.4 s per
subset (timed on one PFC session while preparing); on all 500 subsets of the 128 PFC and CA1
cases that is about 25 hours on one core. It is run here on every 125th subset (4 per session), a
reduction recorded as a deviation.

In [ ]:
AREAS = ["PFC", "CA1"]
VARIANTS = {"PFC": ("PFC", 19), "CA1": ("CA1", 24), "CA1_19": ("CA1", 19)}

def run_readout(counts, z, loc, area, nu, seed, n_subsets, svc_every):
    import wtrack as w
    x, zz = w.process_session(counts, z, loc, area)
    return w.readout_subsets(x, zz, nu, np.random.default_rng(seed), n_subsets, svc_every=svc_every)

t = time.time()
jobs = [(v, ri, si) for v in VARIANTS for ri in range(8) for si in range(w.N_SESSIONS)]
def args(v, ri, si):
    s = D[w.RATS[ri]]["sessions"][si]
    return (s["counts"], s["z"], s["loc"], VARIANTS[v][0], VARIANTS[v][1], [1, list(VARIANTS).index(v), ri, si],
            N_SUBSETS, SVC_EVERY if v != "CA1_19" else 0)
out = Parallel(n_jobs=N_JOBS)(delayed(run_readout)(*args(*j)) for j in jobs)
RO = {v: np.zeros((8, w.N_SESSIONS, N_SUBSETS, 7)) for v in VARIANTS}
for (v, ri, si), o in zip(jobs, out):
    RO[v][ri, si] = o
print(f"readout: {len(jobs)} cases in {time.time() - t:.0f} s")
COLS = ["theory", "hebbian", "dim", "corr", "snf", "ssf", "svc"]
M = {v: {c: (np.nanmean(RO[v][..., k], axis=2)) for k, c in enumerate(COLS)} for v in VARIANTS}   # rat x session
print("non-finite subset values:", {v: int((~np.isfinite(RO[v][..., :6])).sum()) for v in VARIANTS},
      "| SVC subsets per session:", int(np.isfinite(RO["PFC"][0, 0, :, 6]).sum()))
BEH = np.array([[1 - s["frac_correct"] for s in D[r]["sessions"][:w.N_SESSIONS]] for r in w.RATS])   # rat x session
for v in VARIANTS:
    print(f"\n{v}: mean over rats per session (sessions 1 to 8)")
    print(pd.DataFrame({c: M[v][c].mean(0) for c in COLS}, index=pd.Index(range(1, 9), name="session")).round(4).to_string())

### Comparison with the paper's own results

The paper's repository holds the output of its readout script, `results/PFC_results_nu_dt_0.5_ns_500_P_300.npy`
and `results/CA1_results_nu_dt_0.5_ns_500_P_300.npy` (500 subsets x 8 rats x 8 sessions; errors as
behaviour fraction correct, empirical, theory, SVC; geometry as dimension, correlation, SNF, SSF,
...), with 19 units in both areas. They are read here from GitHub at commit 9674a6c, in memory, for
comparison only: no outcome is decided from them. The rat order of that file is the order of the
code's `glob`, not known, so it is compared through the means over rats per session, and the per-rat
behavioural error, which needs no model, is matched to identify the order.

In [ ]:
import io, requests
REPO = "https://raw.githubusercontent.com/awakhloo/population_geometry_optimal_coding/9674a6cb0c39926d70746024ce17de35f87df8f9/results/"
PAPER = {}
for a in AREAS:
    r = requests.get(REPO + f"{a}_results_nu_dt_0.5_ns_500_P_300.npy", timeout=120)
    PAPER[a] = np.load(io.BytesIO(r.content), allow_pickle=True).item()
    print(a, "errs", PAPER[a]["errs"].shape, "geoms", PAPER[a]["all_geoms"].shape)
# match the paper's rats to ours by the behavioural error of 8 sessions, which has no random part
pb = 1 - PAPER["PFC"]["errs"].mean(0)[:, :8, 0]
order = [int(np.argmin(np.abs(pb - BEH[i]).sum(1))) for i in range(8)]
print("paper rat index for", dict(zip(w.RATS, order)), "| largest behavioural error difference after matching", float(np.abs(pb[order] - BEH).max()))
for a, v in [("PFC", "PFC"), ("CA1", "CA1_19")]:
    pe, pg = PAPER[a]["errs"].mean(0)[order][:, :8], PAPER[a]["all_geoms"].mean(0)[order][:, :8]
    comp = {"theory": pe[..., 2], "hebbian": pe[..., 1], "dim": pg[..., 0], "corr": pg[..., 1], "snf": pg[..., 2], "ssf": pg[..., 3]}
    print(f"\n{a}, 19 units: per rat and session, this attempt against the paper's file")
    for k, g in comp.items():
        ours = M[v][k]
        print(f"  {k:8s} r = {np.corrcoef(ours.ravel(), g.ravel())[0, 1]:.3f}, median |relative difference| = {np.median(np.abs(ours - g) / np.abs(g)):.4f}; "
              f"means per session paper {np.round(g.mean(0), 4)} ours {np.round(ours.mean(0), 4)}")
    th, em = pe[..., 2].ravel(), pe[..., 1].ravel()
    q = em <= np.quantile(em, 0.25)
    print(f"  paper's file, fig7f measure: r = {np.corrcoef(th, em)[0, 1]:.3f}, mean theory - empirical = {np.mean(th - em):.4f}, "
          f"mean |theory - empirical| / mean empirical = {np.mean(np.abs(th - em)) / em.mean():.4f}, lowest-quarter theory - empirical = {np.mean(th[q] - em[q]):.4f}")
    print(f"  paper's file, behavioural error against theory r2 = {np.corrcoef(1 - pe[..., 0].ravel(), th)[0, 1] ** 2:.3f}")

## GLM fits (shared pipeline)

Per-unit Poisson GLMs (Methods, Generalized linear model analysis; code `fit-GLMs-rat.py` with the
basis the code chose: position bins of 10 cm, velocity bins of 3 cm/s, variances 100 and 40, centres
from bins holding more than 20 time bins over all of a rat's sessions). Bins are those with the
dataset's speed column above 4 cm/s, as the GLM code (`rat_utils.get_all_lat`) does. The l2 penalty is
chosen per unit by 10-fold cross-validation over 10 values; the folds are scikit-learn's unshuffled
`KFold`, which `GridSearchCV` uses for a regressor, so they involve no random choice. The basis grid
search itself was not repeated: the code's chosen basis is used.

In [ ]:
CENTRES = {}
for ri, r in enumerate(w.RATS):
    flat = np.concatenate([w.glm_session_data(s["counts"], s["z"], s["loc"], "PFC")[1] for s in D[r]["sessions"][:w.N_SESSIONS]])
    CENTRES[r] = w.basis_centres(flat)
    print(r, "basis centres: position", len(CENTRES[r][0]), "vx", len(CENTRES[r][1]), "vy", len(CENTRES[r][2]))

def run_glm(counts, z, loc, area, centres):
    import wtrack as w
    return w.glm_session(counts, z, loc, area, centres)

def gargs(a, ri, si):
    s = D[w.RATS[ri]]["sessions"][si]
    loc = s["loc"]
    if GLM_MAX_UNITS:   # development only; None in the deposited notebook
        idx = np.where(loc == a)[0][:GLM_MAX_UNITS]
        loc = np.where(np.isin(np.arange(len(loc)), idx), loc, "none")
    return s["counts"], s["z"], loc, a, CENTRES[w.RATS[ri]]

t = time.time()
gjobs = [(a, ri, si) for a in AREAS for ri in range(8) for si in range(w.N_SESSIONS)]
gout = Parallel(n_jobs=N_JOBS)(delayed(run_glm)(*gargs(*j)) for j in gjobs)
GLM = {j: o for j, o in zip(gjobs, gout)}
print(f"GLM: {sum(o['x'].shape[1] for o in gout)} unit fits in {time.time() - t:.0f} s")
print("covariates per session:", sorted({o['n_covariates'] for o in gout})[:3], "...", "constant covariates (dropped by nothing; nan if any):", sum(o['n_constant_covariates'] for o in gout))

GS = {}
for (a, ri, si), o in GLM.items():
    for nu in (19, 24):
        if a == "PFC" and nu == 24:
            continue
        rng = np.random.default_rng([2, AREAS.index(a), ri, si, nu])
        GS[(a, nu, ri, si)] = w.glm_summary(o, nu, rng, N_PR)
TD = {(a, nu): np.array([[GS[(a, nu, ri, si)]["task_dim"] for si in range(8)] for ri in range(8)]) for a, nu in [("PFC", 19), ("CA1", 19), ("CA1", 24)]}
for a in AREAS:
    G = pd.DataFrame([dict(rat=w.RATS[ri], session=si + 1, **GS[(a, 19, ri, si)]) for ri in range(8) for si in range(8)])
    print(f"\n{a}: units fitted {G.n_units.sum()}, replaced by the null model {G.n_null.sum()} ({G.n_null.sum() / G.n_units.sum():.1%})")
    print(G.pivot(index="rat", columns="session", values="task_dim").round(2).to_string())

In [ ]:
def s4_s1(g):   # rat x session -> mean over rats at sessions 1, 4, 8
    m = g.mean(0)
    return m[0], m[3], m[7]

def per_rat(g, name):
    return pd.DataFrame(g, index=pd.Index(w.RATS, name="rat"), columns=range(1, 9)).round(4).to_string()

def sem(g):
    return g.std(0, ddof=1) / np.sqrt(g.shape[0])

## fig7d-behavioral-error-learning

Behavioural error decreases quickly over the first sessions and largely plateaus after session four.

> Behavioral error on the continuous alternation task and neural readout error quickly decrease over the first few sessions ... After session four, both behavioral and readout error largely plateau

In [ ]:
print("behavioural error (1 - fraction of trials correct) per rat and session:")
print(per_rat(BEH, "beh"))
m = BEH.mean(0)
print("\nmean over rats per session:", np.round(m, 3), "\ns.e.m.:", np.round(sem(BEH), 3))
e1, e4, e8 = s4_s1(BEH)
print(f"session 1 {e1:.3f}, session 4 {e4:.3f}, session 8 {e8:.3f}; drop 1 to 4 = {e1 - e4:.3f}; |change 4 to 8| = {abs(e8 - e4):.3f}")
print("criterion: session 4 below session 1:", e4 < e1, "| |change 4 to 8| smaller than the drop 1 to 4:", abs(e8 - e4) < (e1 - e4))
print("rats with session 4 below session 1:", int((BEH[:, 3] < BEH[:, 0]).sum()), "of 8")
g, _, p, _ = w.fit_mixed(BEH, quad=False, sessions=slice(0, 4))
print(f"linear mixed model over sessions 1 to 4: slope {g:.4f} per session, P = {p:.2g}")

**Result: reproduced.** The mean behavioural error over the 8 rats falls from 0.668 at session 1 to 0.157 at session 4 (a drop of 0.511, lower in 8 of 8 rats) and is 0.180 at session 8, a change of 0.023 after session 4. The linear mixed model over sessions 1 to 4 has a slope of -0.166 per session (P = 5.8e-18).

## fig7e-readout-error-learning

The task-averaged readout error for position and velocity decreases over learning in PFC and CA1, then plateaus after session four.

> These readout errors decrease over learning (Fig. 7e,f )

In [ ]:
def plateau_check(g):
    e1, e4, e8 = s4_s1(g)
    return dict(s1=round(e1, 4), s4=round(e4, 4), s8=round(e8, 4), drop_1_4=round(e1 - e4, 4), change_4_8=round(e8 - e4, 4),
                holds=bool(e4 < e1 and e8 < e1 and abs(e8 - e4) < (e1 - e4)))
rows = []
for v in VARIANTS:
    for c in ["theory", "hebbian"]:
        rows.append(dict(units=v, reading=c, **plateau_check(M[v][c])))
print(pd.DataFrame(rows).to_string(index=False))
for v in ["PFC", "CA1"]:
    print(f"\n{v} theoretical error per rat and session:")
    print(per_rat(M[v]["theory"], v))
    print("mean", np.round(M[v]["theory"].mean(0), 4), "s.e.m.", np.round(sem(M[v]["theory"]), 4))

**Result: reproduced.** Under the main reading (theoretical error) the mean over rats falls from session 1 to 4 by 0.0459 in PFC (0.3626 to 0.3167) and by 0.0401 in CA1 (0.3523 to 0.3121), and changes by 0.0153 and 0.0119 from session 4 to 8; session 8 is below session 1 in both. The criterion holds in both areas under both readings and with the 19-unit CA1 variant. In both areas the error rises again at session 8 (PFC 0.3320, CA1 0.3241).

## figs12c-behavior-readout-r2

Single-session readout errors correlate with behavioural error, R2 = 0.48 in CA1 and 0.51 in PFC.

> are correlated with single-session behavioral task performance ( R 2 = 0.48 (CA1) and R 2 = 0.51 (PFC); Supplementary Fig. 12c )

In [ ]:
rows = []
for v in VARIANTS:
    for c in ["theory", "hebbian"]:
        r, p = stats.pearsonr(BEH.ravel(), M[v][c].ravel())
        rows.append(dict(units=v, reading=c, r=round(r, 3), r2=round(r * r, 3), p=p, n=BEH.size))
R = pd.DataFrame(rows)
print(R.to_string(index=False))
print("expected: CA1 0.48 +- 0.048 (0.432 to 0.528), PFC 0.51 +- 0.051 (0.459 to 0.561)")
for c in ["theory", "hebbian"]:
    ca1 = R[(R.units == "CA1") & (R.reading == c)].r2.item(); pfc = R[(R.units == "PFC") & (R.reading == c)].r2.item()
    print(f"{c}: CA1 within {abs(ca1 - 0.48) <= 0.048}, PFC within {abs(pfc - 0.51) <= 0.051}")
print("\nper rat (8 sessions each), r2 of behavioural error against theoretical error:")
print(pd.DataFrame({v: [stats.pearsonr(BEH[i], M[v]['theory'][i])[0] ** 2 for i in range(8)] for v in ["PFC", "CA1"]}, index=w.RATS).round(3).to_string())
fig, ax = plt.subplots(1, 2, figsize=(8, 3.5))
for a, v in zip(ax, ["CA1", "PFC"]):
    a.scatter(BEH.ravel(), M[v]["theory"].ravel(), s=12)
    a.set_xlabel("behavioural error"); a.set_ylabel("theoretical readout error"); a.set_title(v)
plt.tight_layout(); plt.show()
print("plotted ranges: behavioural error", BEH.min().round(3), "to", BEH.max().round(3), "; theory PFC", M["PFC"]["theory"].min().round(3), "to", M["PFC"]["theory"].max().round(3),
      "CA1", M["CA1"]["theory"].min().round(3), "to", M["CA1"]["theory"].max().round(3), "; no points outside the axes")

**Result: reproduced.** Over the 64 rat and session points the squared correlation of behavioural error with the theoretical error is 0.493 in CA1 and 0.508 in PFC, both within tolerance (0.48 +- 0.048, 0.51 +- 0.051). With the empirical Hebbian error (second reading) CA1 is 0.579, outside, and PFC 0.536, inside; with 19 CA1 units the theoretical reading gives 0.475.

## fig7f-theory-vs-empirical

The theoretical error agrees well overall with the empirical Hebbian error, with a small bias at low error values.

> Empirical generalization error versus our estimate from equation ( 1 ). Although there is a small bias for low error values, we find a good agreement overall

Criterion, as the claims file now has it after the question this attempt asked (`questions/wakhloo-2026.yaml`): per area, over the 64 rat and session pairs of means over the 500 subsets, the Pearson correlation is positive with p < 0.05 and r is at least 0.9. The earlier criterion also asked for a mean absolute difference below 10% of the mean empirical error; the reader found that this bound fails on the paper's own figure, which the legend calls good, and dropped it. The signed and absolute differences and the difference in the lowest-error quarter are printed and not judged, as the legend allows a bias.

In [ ]:
rows = []
for v in VARIANTS:
    th, em = M[v]["theory"].ravel(), M[v]["hebbian"].ravel()
    r, p = stats.pearsonr(th, em)
    q = em <= np.quantile(em, 0.25)
    rows.append(dict(units=v, r=round(r, 3), p=p, mean_abs_diff=round(np.mean(np.abs(th - em)), 4), mean_emp=round(em.mean(), 4),
                     ratio=round(np.mean(np.abs(th - em)) / em.mean(), 4), mean_diff_theory_minus_emp=round(np.mean(th - em), 4),
                     diff_lowest_quarter=round(np.mean(th[q] - em[q]), 4), diff_top_quarter=round(np.mean(th[em >= np.quantile(em, 0.75)] - em[em >= np.quantile(em, 0.75)]), 4),
                     holds=bool(r > 0 and p < 0.05 and r >= 0.9)))
# criterion (claims file, as amended after the question of this attempt): per area, r > 0 with p < 0.05 and r >= 0.9;
# the signed and absolute differences and the lowest-quarter difference are reported, not judged (the legend allows a bias)
print(pd.DataFrame(rows).to_string(index=False))
for a, (v1, v2) in {"PFC": ("PFC", "PFC"), "CA1": ("CA1", "CA1_19")}.items():
    print(a, "holds (main units):", [x["holds"] for x in rows if x["units"] == v1][0], "| holds (19 units):", [x["holds"] for x in rows if x["units"] == v2][0])
print("subset-level: per subset, mean |theory - Hebbian| =", {v: round(float(np.mean(np.abs(RO[v][..., 0] - RO[v][..., 1]))), 4) for v in VARIANTS})

**Result: reproduced.** The theoretical and empirical errors are strongly correlated in both areas, r = 0.983 in PFC and 0.948 in CA1 (24 units), P below 1e-31, both above the criterion's 0.9, and the 19-unit CA1 variant (the code's) holds as well. The theory lies above the empirical error on average (by 0.034 in PFC and 0.031 in CA1), more so in the lowest-error quarter (0.048, 0.044) than in the highest (0.015, 0.016), which is the bias at low error that the legend describes; the mean absolute difference is 11.55% (PFC) and 10.69% (CA1) of the mean empirical error, close to the 11.6% and 9.0% of the paper's own results file (comparison cell). Under the claims file before the reader's answer, which also asked for a difference below 10%, the outcome was not reproduced.

## fig7g-total-dimension-drops-early

The total neural dimension in PFC and CA1 drops over the first four sessions.

> Interestingly, we find that although the total neural dimension drops over these initial sessions (Fig. 7g )

In [ ]:
for v in VARIANTS:
    g = M[v]["dim"]
    d1, d4, d8 = s4_s1(g)
    sl, _, p, _ = w.fit_mixed(g, quad=False, sessions=slice(0, 4))
    print(f"{v}: session 1 {d1:.3f}, session 4 {d4:.3f}, session 8 {d8:.3f}; session 4 below 1: {d4 < d1}; rats lower at 4: {int((g[:, 3] < g[:, 0]).sum())} of 8; "
          f"mixed slope 1 to 4 {sl:.3f} (P = {p:.2g})")
    print("   mean per session", np.round(g.mean(0), 3), "s.e.m.", np.round(sem(g), 3))

**Result: reproduced.** The mean total dimension falls from session 1 to 4 in PFC (15.833 to 14.516) and CA1 (19.537 to 18.130), in 8 of 8 rats in each area; the mixed-model slopes over sessions 1 to 4 are -0.426 (P = 1.3e-14) and -0.485 (P = 3.5e-08). The 19-unit CA1 variant also falls (16.027 to 15.128).

## fig7h-task-dimension-increases

The task-relevant dimension from Poisson GLMs increases over learning, including over the first four sessions.

> the dimension of the subspace containing task-relevant information (that is, the dimension of the ‘signal subspace’) increases monotonically over learning, even during those initial four sessions (Fig. 7h )

In [ ]:
for key in [("PFC", 19), ("CA1", 19), ("CA1", 24)]:
    g = TD[key]
    t1, t4, t8 = s4_s1(g)
    mono = bool(np.all(np.diff(g.mean(0)) >= 0))
    print(f"{key[0]} ({key[1]} units): session 1 {t1:.3f}, 4 {t4:.3f}, 8 {t8:.3f}; (a) rises 1 to 4 and 1 to 8: {t4 > t1 and t8 > t1}; "
          f"(b) no fall between consecutive sessions: {mono}; rats higher at 4 than 1: {int((g[:, 3] > g[:, 0]).sum())} of 8")
    print("   mean per session", np.round(g.mean(0), 3), "s.e.m.", np.round(sem(g), 3))
for a in AREAS:
    print(f"\n{a} task-relevant dimension per rat and session (19 units):")
    print(per_rat(TD[(a, 19)], a))

**Result: reproduced** under reading (a). The mean task-relevant dimension rises from session 1 to 4 and to 8 in PFC (7.131, 9.040, 9.543) and CA1 (6.892, 9.134, 9.008); it is higher at session 4 than 1 in 7 of 8 PFC and 8 of 8 CA1 rats. Under reading (b) it is not monotonic in either area: the PFC mean falls from session 6 to 7 and the CA1 mean from 5 to 6 and from 7 to 8. With 24 CA1 units it rises from 7.926 to 10.286 and 10.260.

## fig7hk-measures-increase-early

Over the first four sessions nearly every geometric measure (task dimension, SSF, correlation, SNF) increases.

> Over the first four sessions, both behavioral and neural readout error rapidly improve (Fig. 7d,e ). During this time, nearly every geometric measure increases (Fig. 7h–k )

In [ ]:
for v, td in [("PFC", ("PFC", 19)), ("CA1", ("CA1", 19)), ("CA1_19", ("CA1", 19))]:
    meas = {"task_dim": TD[td], "ssf": M[v]["ssf"], "corr": M[v]["corr"], "snf": M[v]["snf"]}
    up = {k: bool(g.mean(0)[3] > g.mean(0)[0]) for k, g in meas.items()}
    print(f"{v}: increases 1 to 4 {up}; {sum(up.values())} of 4; holds (at least 3): {sum(up.values()) >= 3}")
    print("   session 1 / 4:", {k: (round(g.mean(0)[0], 4), round(g.mean(0)[3], 4)) for k, g in meas.items()})

**Result: reproduced.** From session 1 to 4 all four measures increase in PFC (task dimension 7.131 to 9.040, SSF 0.6323 to 0.6661, correlation 0.0182 to 0.0337, SNF 0.3426 to 0.6985) and three of four in CA1, where SSF falls slightly (0.6905 to 0.6787). The 19-unit CA1 variant gives the same three of four.

## fig7-late-reversal

After session four the total dimension begins to increase while the correlation drops.

> At this point, several of these initial trends reverse; in particular, the total dimension begins to increase, whereas the correlation drops

In [ ]:
for v in VARIANTS:
    d, c = M[v]["dim"], M[v]["corr"]
    a_dim, a_corr = d.mean(0)[7] - d.mean(0)[3], c.mean(0)[7] - c.mean(0)[3]
    sd, _, pd_, _ = w.fit_mixed(d, quad=False, sessions=slice(3, 8))
    sc, _, pc, _ = w.fit_mixed(c, quad=False, sessions=slice(3, 8))
    print(f"{v}: (a) session 8 - 4: dim {a_dim:+.3f}, corr {a_corr:+.5f} -> holds {a_dim > 0 and a_corr < 0}; "
          f"(b) mixed slope 4 to 8: dim {sd:+.3f} (P = {pd_:.2g}), corr {sc:+.5f} (P = {pc:.2g}) -> holds {sd > 0 and sc < 0}")

**Result: reproduced.** From session 4 to 8 the total dimension rises and the correlation falls in both areas under both readings: (a) PFC +0.936 and -0.00764, CA1 +0.784 and -0.00603; (b) mixed slopes over sessions 4 to 8 of +0.197 (P = 7.5e-07) and -0.00154 (P = 7e-05) in PFC, +0.169 (P = 0.0022) and -0.00130 (P = 0.0022) in CA1.

In [ ]:
# Mixed models of the figure notebook: per area, quadratic models of dim, corr, snf, ssf and task dim,
# linear models of ssf and task dim; Benjamini-Hochberg over the 24 P values (14 linear, 10 quadratic).
def mixed_table(units_ro, units_td, t0=0):
    rows = []
    for a in AREAS:
        v = units_ro[a]; td = TD[units_td[a]]
        for meas in ["dim", "corr", "snf", "ssf"]:
            g, e, pg, pe = w.fit_mixed(M[v][meas], quad=True, t0=t0)
            rows.append(dict(area=a, model="quad", meas=meas, gamma=g, p_gamma=pg, eta=e, p_eta=pe))
        g, e, pg, pe = w.fit_mixed(M[v]["ssf"], quad=False, t0=t0)
        rows.append(dict(area=a, model="lin", meas="ssf", gamma=g, p_gamma=pg, eta=e, p_eta=pe))
        g, e, pg, pe = w.fit_mixed(td, quad=True, t0=t0)
        rows.append(dict(area=a, model="quad", meas="task_dim", gamma=g, p_gamma=pg, eta=e, p_eta=pe))
        g, e, pg, pe = w.fit_mixed(td, quad=False, t0=t0)
        rows.append(dict(area=a, model="lin", meas="task_dim", gamma=g, p_gamma=pg, eta=e, p_eta=pe))
    T = pd.DataFrame(rows)
    pv = np.concatenate([T.p_gamma.values, T.p_eta.dropna().values])
    adj = multipletests(pv, method="fdr_bh")[1]
    T["p_gamma_adj"] = adj[:len(T)]
    T["p_eta_adj"] = np.nan
    T.loc[T.p_eta.notna(), "p_eta_adj"] = adj[len(T):]
    return T

MIX = {
    "main (t = 0..7, CA1 24 units, task dim 19)": mixed_table({"PFC": "PFC", "CA1": "CA1"}, {"PFC": ("PFC", 19), "CA1": ("CA1", 19)}),
    "t = 1..8": mixed_table({"PFC": "PFC", "CA1": "CA1"}, {"PFC": ("PFC", 19), "CA1": ("CA1", 19)}, t0=1),
    "CA1 19 units (code)": mixed_table({"PFC": "PFC", "CA1": "CA1_19"}, {"PFC": ("PFC", 19), "CA1": ("CA1", 19)}),
    "CA1 24 units, task dim 24": mixed_table({"PFC": "PFC", "CA1": "CA1"}, {"PFC": ("PFC", 19), "CA1": ("CA1", 24)}),
}
fmt = {c: "{:.3g}".format for c in ["gamma", "p_gamma", "eta", "p_eta", "p_gamma_adj", "p_eta_adj"]}
for k, T in MIX.items():
    print("=== ", k)
    print(T.to_string(index=False, formatters=fmt))

## mixed-quadratic-dim-corr-snf

Quadratic mixed models of total dimension, correlation and SNF have linear and quadratic terms with P < 10^-5 in both areas, of opposite sign.

> We found strong, significant effects for both the linear and quadratic terms in PFC and CA1 for the total dimension, correlation and SNF ( P < 10 −5 ; Supplementary Table 2 ). In each case, the sign of the quadratic term was the opposite of the linear term

In [ ]:
for k, T in MIX.items():
    q = T[(T.model == "quad") & T.meas.isin(["dim", "corr", "snf"])]
    opp = bool(np.all(np.sign(q.gamma) != np.sign(q.eta)))
    for adj in ["raw", "adjusted"]:
        P = np.concatenate([q.p_gamma, q.p_eta]) if adj == "raw" else np.concatenate([q.p_gamma_adj, q.p_eta_adj])
        print(f"{k} | {adj}: P below 1e-5 {int((P < 1e-5).sum())} of 12, below 0.05 {int((P < 0.05).sum())} of 12, largest P {P.max():.2g}; opposite signs in all six: {opp}")

**Result: reproduced.** In the main reading all 12 P values (two areas, three measures, linear and quadratic terms) are below 1e-5, the largest 7.2e-07 raw and 1.2e-06 adjusted, and the quadratic term is opposite in sign to the linear term in all six models. The same holds with sessions coded 1 to 8 and with 19 CA1 units.

## mixed-quadratic-ssf-not-nonmonotonic

The quadratic mixed model gives no evidence of a non-monotonic SSF trend in either area (P > 0.05).

> We did not find evidence for a non-monotonic trend in the SSF in either region ( P > 0.05)

In [ ]:
for k, T in MIX.items():
    q = T[(T.model == "quad") & (T.meas == "ssf")]
    print(k, "|", "; ".join(f"{r.area}: eta {r.eta:.3g}, P {r.p_eta:.3g} (adj. {r.p_eta_adj:.3g})" for r in q.itertuples()),
          "| raw P > 0.05 in both:", bool(np.all(q.p_eta > 0.05)))

**Result: reproduced.** The quadratic term of the SSF model is not significant in either area: PFC eta = -0.00115 (P = 0.366, adjusted 0.382), CA1 eta = 0.00121 (P = 0.313, adjusted 0.342). The 19-unit CA1 variant gives P = 0.419.

## mixed-quadratic-task-dim

The quadratic mixed model has weaker but significant effects for the task-relevant dimension (P < 0.01).

> Although we did find weaker significant effects of the quadratic model for the task-relevant dimension ( P < 0.01), the model was a comparatively poorer fit to this quantity in PFC

In [ ]:
for k, T in MIX.items():
    q = T[(T.model == "quad") & (T.meas == "task_dim")]
    print(k, "|", "; ".join(f"{r.area}: gamma {r.gamma:.3g} (P {r.p_gamma:.2g}, adj. {r.p_gamma_adj:.2g}), eta {r.eta:.3g} (P {r.p_eta:.2g}, adj. {r.p_eta_adj:.2g})" for r in q.itertuples()))
    print("    raw: both terms P < 0.01 in both areas:", bool(np.all(q.p_gamma < 0.01) and np.all(q.p_eta < 0.01)),
          "| adjusted:", bool(np.all(q.p_gamma_adj < 0.01) and np.all(q.p_eta_adj < 0.01)))

**Result: reproduced.** Both terms of the quadratic model of the task-relevant dimension have P < 0.01 in both areas, raw and adjusted: PFC gamma = 0.714 (P = 7.9e-06), eta = -0.0664 (P = 0.0025); CA1 gamma = 1.01 (P = 5.5e-11), eta = -0.109 (P = 2.3e-07).

## mixed-linear-ssf-task-dim

Linear mixed models show monotonic effects on SSF (P < 10^-3) and task-relevant dimension (P < 10^-5) in both areas.

> which demonstrated significant, monotonic effects of learning on the SSF and task-relevant dimension across both regions ( P < 10 −3 SSF and P < 10 −5 task dimension; Supplementary Table 2 )

In [ ]:
for k, T in MIX.items():
    l = T[T.model == "lin"]
    print(k, "|", "; ".join(f"{r.area} {r.meas}: gamma {r.gamma:.3g}, P {r.p_gamma:.2g} (adj. {r.p_gamma_adj:.2g})" for r in l.itertuples()))
    for adj in ["p_gamma", "p_gamma_adj"]:
        ssf = l[l.meas == "ssf"]; td = l[l.meas == "task_dim"]
        ok = bool(np.all(ssf.gamma > 0) and np.all(ssf[adj] < 1e-3) and np.all(td.gamma > 0) and np.all(td[adj] < 1e-5))
        part = bool(np.all(l.gamma > 0) and np.all(l[adj] < 0.05))
        print(f"    {adj}: full criterion {ok}; signs positive with P < 0.05 {part}")

**Result: partially reproduced.** All four slopes are positive with P < 0.05. PFC SSF (0.0119, P = 3.0e-06), PFC task dimension (0.249, P = 1.2e-07) and CA1 task dimension (0.242, P = 2.1e-06) meet their bounds; CA1 SSF (0.00787) has P = 0.00106 raw and 0.00134 adjusted, just above the 1e-3 bound. With the code's 19 CA1 units it is P = 0.00065 and the full criterion holds, so the shortfall rests on the unit count and on random subsets near the bound.

## late-trends-match-optimal

From session 4 to 8 the dimension measures and SSF increase while the correlation decreases.

> Once the readout and behavioral errors begin to plateau at session four, we find that the ensuing trends are consistent with our predictions for optimal representations ... the dimensionality measures and SSF increase over learning, whereas the neural–latent correlation decreases

In [ ]:
for v, td in [("PFC", ("PFC", 19)), ("CA1", ("CA1", 19)), ("CA1_19", ("CA1", 19))]:
    meas = {"dim": M[v]["dim"], "task_dim": TD[td], "ssf": M[v]["ssf"], "corr": M[v]["corr"]}
    a = {k: g.mean(0)[7] - g.mean(0)[3] for k, g in meas.items()}
    b = {k: w.fit_mixed(g, quad=False, sessions=slice(3, 8))[::2] for k, g in meas.items()}
    ha = a["dim"] > 0 and a["task_dim"] > 0 and a["ssf"] > 0 and a["corr"] < 0
    hb = b["dim"][0] > 0 and b["task_dim"][0] > 0 and b["ssf"][0] > 0 and b["corr"][0] < 0
    print(f"{v}: (a) session 8 - 4 {dict((k, round(x, 5)) for k, x in a.items())} -> {ha}")
    print(f"     (b) mixed slope 4 to 8 (slope, P) {dict((k, (round(x[0], 5), float('%.2g' % x[1]))) for k, x in b.items())} -> {hb}")

**Result: partially reproduced.** In PFC, from session 4 to 8, both dimension measures and SSF rise and the correlation falls under both readings (total dimension +0.936, task dimension +0.502, SSF +0.0472, correlation -0.00764; mixed slopes all of the same signs). In CA1 the total dimension (+0.784), SSF (+0.0531) and correlation (-0.00603) follow the prediction, but the task-relevant dimension falls (-0.126; slope -0.015, P = 0.82), so the criterion fails there.

## glm-cross-validated-d2

Average cross-validated population D2 of 0.29 (PFC) and 0.25 (CA1); s.d. across rats of session-averaged D2 0.022 (PFC) and 0.1 (CA1).

> The average cross-validated population D 2 across all sessions and rats was 0.29 in PFC and 0.25 in CA1, and the standard deviation of the session-averaged population D 2 across rats was 0.022 in PFC and 0.1 in CA1

In [ ]:
exp = {"PFC": (0.29, 0.022), "CA1": (0.25, 0.1)}
# readings: (i) each unit's own best score, session value the mean over units as the code (nan if a unit
# has no valid score), then nan-mean over sessions as rat_utils.find_best_params; (ii) the same with a
# nan-mean over units; (iii) units replaced by the null model counted as 0
for a in AREAS:
    for key in ["d2", "d2_unit_nanmean", "d2_null0"]:
        g = np.array([[GS[(a, 19, ri, si)][key] for si in range(8)] for ri in range(8)])
        rm = np.nanmean(g, 1)
        m = np.nanmean(g)
        ok = [abs(m - exp[a][0]) <= exp[a][0] / 10, abs(rm.std(ddof=0) - exp[a][1]) <= exp[a][1] / 10, abs(rm.std(ddof=1) - exp[a][1]) <= exp[a][1] / 10]
        print(f"{a} {key}: mean over sessions = {m:.4f} (expected {exp[a][0]} +- {exp[a][0] / 10:.3f}, within {ok[0]}); nan sessions {int(np.isnan(g).sum())}; "
              f"s.d. over rats of rat means ddof 0 = {rm.std(ddof=0):.4f} (within {ok[1]}), ddof 1 = {rm.std(ddof=1):.4f} (within {ok[2]}) (expected {exp[a][1]} +- {exp[a][1] / 10:.4f})")
        print("    per rat", {r: round(float(x), 3) for r, x in zip(w.RATS, rm)})
    allbest = np.concatenate([GLM[(a, ri, si)]["scores"].max(1) for ri in range(8) for si in range(8)])
    print(f"    {a}: unit best scores, median {np.nanmedian(allbest):.3f}, 10th to 90th percentile {np.nanpercentile(allbest, 10):.3f} to {np.nanpercentile(allbest, 90):.3f}, "
          f"nan {int(np.isnan(allbest).sum())}; best alpha at the grid's edges: "
          f"{sum(int((np.nanargmax(np.where(np.isfinite(GLM[(a, ri, si)]['scores']), GLM[(a, ri, si)]['scores'], -np.inf), 1) == k).sum()) for ri in range(8) for si in range(8) for k in (0, 9))} units")

**Result: partially reproduced.** Three of the four values match under the first reading (each unit's own best score, the code's mean): the mean population D2 is 0.279 in PFC (0.29 +- 0.029) and 0.245 in CA1 (0.25 +- 0.025), and the standard deviation over rats of the session-averaged D2 is 0.1005 in CA1 (ddof 0; 0.1075 with ddof 1; 0.1 +- 0.01). In PFC it is 0.0599 (ddof 0) or 0.0640 (ddof 1), nearly three times the reported 0.022, and no reading brings it within tolerance (0.057 to 0.066); the per-rat PFC means run from 0.161 (KL8) to 0.336 (ZT2). Counting null-model units as 0 moves CA1 out of tolerance (0.277). One PFC unit (JS21) has no valid score; the nan-mean over units gives 0.281. For 1202 of 2327 PFC and 1750 of 3357 CA1 units the chosen penalty is at an end of the code's grid.

## svc-hebbian-correlation

The Hebbian error is strongly correlated with the error of a linear SVC on the same tasks (tested here in PFC and CA1).

> Across every analysis reported in this paper, we found a strong correlation between SVC and Hebbian errors, particularly in the brain data

In [ ]:
for v in ["PFC", "CA1"]:
    svc = M[v]["svc"].ravel()
    # the theoretical and Hebbian errors on the same 4 subsets as the SVC, and on all 500 subsets
    sel = np.isfinite(RO[v][0, 0, :, 6])
    th4 = RO[v][:, :, sel, 0].mean(2).ravel(); he4 = RO[v][:, :, sel, 1].mean(2).ravel()
    for name, x in [("theory, all subsets", M[v]["theory"].ravel()), ("theory, SVC subsets", th4),
                    ("Hebbian, all subsets", M[v]["hebbian"].ravel()), ("Hebbian, SVC subsets", he4)]:
        r, p = stats.pearsonr(x, svc)
        print(f"{v} {name}: r = {r:.3f}, r2 = {r * r:.3f}, P = {p:.2g}; (a) r > 0 and P < 0.05: {r > 0 and p < 0.05}; (b) r2 >= 0.5: {r * r >= 0.5}")
    print(f"    {v}: SVC error {svc.min():.3f} to {svc.max():.3f}, mean {svc.mean():.3f}; Hebbian mean {M[v]['hebbian'].mean():.3f}; SVC subsets per session {int(sel.sum())}")

**Result: reproduced.** Over the 64 rat and session values the squared correlation between the theoretical Hebbian error and the SVC error is 0.926 in PFC and 0.832 in CA1 (P below 1e-24), so both readings hold in both areas; with the empirical Hebbian error it is 0.943 and 0.946. The SVC was run on 4 of the 500 subsets per session; on those same subsets the theoretical error gives 0.931 and 0.829.

## Summary

| Claim | Outcome |
|---|---|
| fig7d-behavioral-error-learning | reproduced |
| fig7e-readout-error-learning | reproduced |
| figs12c-behavior-readout-r2 | reproduced (0.493 CA1, 0.508 PFC) |
| fig7f-theory-vs-empirical | reproduced (r 0.98 and 0.95; criterion amended after a question, earlier not reproduced) |
| fig7g-total-dimension-drops-early | reproduced |
| fig7h-task-dimension-increases | reproduced (reading a) |
| fig7hk-measures-increase-early | reproduced |
| fig7-late-reversal | reproduced |
| mixed-quadratic-dim-corr-snf | reproduced |
| mixed-quadratic-ssf-not-nonmonotonic | reproduced |
| mixed-quadratic-task-dim | reproduced |
| mixed-linear-ssf-task-dim | partially reproduced (CA1 SSF P = 0.00106 against 1e-3) |
| late-trends-match-optimal | partially reproduced (PFC only) |
| glm-cross-validated-d2 | partially reproduced (PFC s.d. across rats 0.060 against 0.022) |
| svc-hebbian-correlation | reproduced (PFC and CA1 part) |
| theory, simulations, pose network and macaque claims (12) | blocked |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

How the analysis differs from the paper's: the pipeline is the paper's code, re-implemented on the
streamed files (h5py in place of pynwb; Nadaraya-Watson smoothing truncated at ten bandwidths); the
random subsets use their own seeds; CA1 uses 24 units as the Methods say, with the code's 19 as a
variant; the linear SVC is run on 4 of the 500 subsets per session; the GLM basis is the code's
chosen one, without repeating its grid search. The per-rat behavioural errors equal those of the
paper's results file, and the readout measures agree with it closely (comparison cell).

What was found about the dataset: the eight rats and the 8 sessions each that the paper uses are all
there, with the units, areas, trials (correct, trajectory type) and position the analysis needs, and
the fewest units per session (19 PFC, 24 CA1) are the numbers the Methods use. Both ZT2 files carry
`session_id` "01"; nothing in the files says that they are two parts of one rat's day or in which
order, which the code supplies. Position is sampled at 27 to 33 frames per second (the Methods say
30). JS15's timestamps run from 3605 to 90718 s, a span of about 24 hours for a single-day dataset.
Sessions last 13 to 20 minutes. Trials within a session are contiguous, with no gaps.

In [ ]:
print(f"total run time {(time.time() - T0) / 60:.1f} min; peak memory {resource.getrusage(resource.RUSAGE_SELF).ru_maxrss / 1e6:.2f} GB")